In [1]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())

import os
import json
import torch

from config import RESULTS_DIR, SEEDS, DEVICE
from datasets import get_dataloaders
from models import get_model
from few_shot import train_protonet, evaluate_fixed, create_fixed_episodes
from utils import set_seed
from early_stopping import EarlyStopping

from analysis import *
from experiment_runner import run_all_experiments 


In [2]:
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")


def run_protonet_experiment(model_name, k_shot=5):

    results = []

    for seed in SEEDS:
        print(f"Model={model_name}, seed={seed}")
        set_seed(seed)

        train_loader, val_loader, test_loader = get_dataloaders(
            batch_size=64,
            few_shot_k=20,
            model_name=model_name
        )

        train_dataset = train_loader.dataset
        val_dataset = val_loader.dataset
        test_dataset = test_loader.dataset

        model = get_model(model_name, dropout=0.1, for_protonet=True)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=100
        )
        path=f"results/exp3_fewshot/{model_name}_seed{seed}.pth"
        logs = train_protonet(
            model,
            train_dataset,
            val_dataset,
            optimizer,
            device,
            epochs=20,
            n_way=10,
            k_shot=k_shot,
            q_query=5,
            episodes_per_epoch=100 ,
            scheduler=scheduler
        )


        test_episodes = create_fixed_episodes(
        test_dataset, n_way=10, k_shot=5, q_query=5, n_episodes=200
        )

        test_loss, test_acc = evaluate_fixed(
            model, test_dataset, test_episodes, device, n_way=10
        )
        print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

        results.append({
            "test_acc": test_acc,
            "logs": logs
        })

    return results

In [3]:
df = load_results("./results/exp1_hparams")
summary = summarize(df, ["model", "lr", "weight_decay", "dropout", "batch_size"])
config_keys = ["model", "lr", "weight_decay", "dropout", "batch_size"]

best_cnn = get_best_configs(summary, top_k=10, model="cnn", metric="mean_test_macro_f1")
best_cnn_dict = best_cnn.iloc[0][config_keys].to_dict()
best_resnet = get_best_configs(summary, top_k=10, model="resnet18", metric="mean_test_macro_f1")
best_resnet_dict = best_resnet.iloc[0][config_keys].to_dict()

best_cnn_dict['few_shot_k'] = 20
best_resnet_dict['few_shot_k'] = 20

config = [best_cnn_dict, best_resnet_dict]

In [4]:
run_all_experiments(
    folder="exp3_fewshot/classic",
    configs_list=config,
)

Seed: 0, Config: {"model": "cnn", "lr": 0.00046082521043368826, "weight_decay": 3.600654215799517e-05, "dropout": 0.3, "batch_size": 64, "few_shot_k": 20}
Epoch 1: train=2.3192, val=2.3027, acc=0.1000
Epoch 2: train=2.2529, val=2.3033, acc=0.1000
Epoch 3: train=2.1935, val=2.2955, acc=0.1187
Epoch 4: train=2.1211, val=2.2431, acc=0.1418
Epoch 5: train=2.1107, val=2.1981, acc=0.1464
Epoch 6: train=2.0483, val=2.1231, acc=0.1893
Epoch 7: train=2.0790, val=2.0594, acc=0.2260
Epoch 8: train=2.0226, val=2.0215, acc=0.2551
Epoch 9: train=2.0013, val=2.1646, acc=0.2047
Epoch 10: train=2.0065, val=2.2103, acc=0.2257
Epoch 11: train=1.9663, val=2.4086, acc=0.2222
Epoch 12: train=1.9752, val=2.1283, acc=0.2532
Epoch 13: train=1.9400, val=2.0397, acc=0.2697
Epoch 14: train=1.9191, val=2.0144, acc=0.2677
Epoch 15: train=1.8625, val=1.9870, acc=0.2728
Epoch 16: train=1.9373, val=1.9947, acc=0.2786
Epoch 17: train=1.8532, val=2.0410, acc=0.2659
Epoch 18: train=1.8871, val=1.9881, acc=0.2821
Epoch 19

In [5]:
all_results = []

for model in ["resnet18","cnn"]:
    for k in [5,10,15]:
        res = run_protonet_experiment(model, k_shot=k)
        all_results.append({
            "model": model,
            "k_shot": k,
            "runs": res
        })

with open("results/exp3_fewshot/protonet.json", "w") as f:
    json.dump(all_results, f, indent=2)

Model=resnet18, seed=0
Epoch 1: train_loss=1.6068, train_acc=0.8220 | val_loss=1.8166, val_acc=0.4723
Epoch 2: train_loss=1.3914, train_acc=1.0000 | val_loss=1.8673, val_acc=0.5877
Epoch 3: train_loss=1.3538, train_acc=1.0000 | val_loss=1.8634, val_acc=0.5923
Epoch 4: train_loss=1.3423, train_acc=1.0000 | val_loss=1.8548, val_acc=0.5922
Epoch 5: train_loss=1.3319, train_acc=1.0000 | val_loss=1.8483, val_acc=0.5958
Epoch 6: train_loss=1.3215, train_acc=1.0000 | val_loss=1.8449, val_acc=0.5934
Epoch 7: train_loss=1.3111, train_acc=1.0000 | val_loss=1.8380, val_acc=0.5931
Epoch 8: train_loss=1.3006, train_acc=1.0000 | val_loss=1.8333, val_acc=0.5934
Epoch 9: train_loss=1.2899, train_acc=1.0000 | val_loss=1.8286, val_acc=0.5924
Epoch 10: train_loss=1.2791, train_acc=1.0000 | val_loss=1.8235, val_acc=0.5904
Epoch 11: train_loss=1.2680, train_acc=1.0000 | val_loss=1.8178, val_acc=0.5903
Epoch 12: train_loss=1.2568, train_acc=1.0000 | val_loss=1.8119, val_acc=0.5906
Epoch 13: train_loss=1.245